# 01 · EDA on the frozen master table

Run `python src/build_master.py` first. This notebook only **reads** `data/master_v1.csv` (never rebuilds it) and makes the EDA tables/figures for the report.

*Edited from the midterm EDA notebook (kept in `notebooks/archive_midterm/`).*

In [ ]:
import os, sys, warnings
from pathlib import Path
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
import config as C
warnings.filterwarnings("ignore")
%matplotlib inline
pd.set_option("display.width", 140); pd.set_option("display.max_columns", 20)

MASTER = os.environ.get("MASTER_PATH", str(C.MASTER_PATH))   # set MASTER_PATH to test on fake data
OUT = os.environ.get("EDA_OUT", str(C.OUT_DIR / "eda")); os.makedirs(OUT, exist_ok=True)
VIX_STRESS = C.VIX_STRESS
NAVY, GREY, ACCENT, LIGHT = "#0f2d6b", "#9aa5b1", "#b24a28", "#c9d2d9"
SOURCE_NOTE = "Source: Yahoo Finance via yfinance (^GSPC, ^VIX, top 50 S&P 500 stocks)"
def style(ax): ax.spines[["top", "right"]].set_visible(False)
def title(ax, text, size=12): ax.set_title(text, fontsize=size, fontweight="bold", loc="left")

In [ ]:
from fincare_common import load_master
data = load_master(MASTER)
top = [c[:-4] for c in data.columns if c.endswith("_ret") and c != "ret_index"]
ret = data["ret_index"]
stock_rets = data[[f"{t}_ret" for t in top]].rename(columns=lambda c: c[:-4])
# index level for Figure 1 and the ADF test: rebuilt from returns (starts at 100)
data["SP500"] = 100 * (1 + ret).cumprod()
print(f"MASTER: {data.shape[0]:,} rows x {data.shape[1]} columns, {data.index.min().date()} to {data.index.max().date()}; {len(top)} stocks")

## 8. Table 3: Summary statistics


In [ ]:
table3 = pd.DataFrame(
    [
        ("Trading days (final)", f"{len(data):,}"),
        ("Period", f"{data.index.min().date()} to {data.index.max().date()}"),
        ("Columns (final)", f"{data.shape[1]:,}"),
        ("Index daily return: mean", f"{ret.mean() * 100:.3f}%"),
        ("Index daily return: std", f"{ret.std() * 100:.3f}%"),
        (
            "Index daily return: min / max",
            f"{ret.min() * 100:.2f}% / {ret.max() * 100:.2f}%",
        ),
        ("Share of up days", f"{(ret > 0).mean() * 100:.1f}%"),
        (
            "Mean / std of the 50 stock returns (avg across stocks)",
            f"{stock_rets.mean().mean() * 100:.3f}% / {stock_rets.std().mean() * 100:.3f}%",
        ),
        (
            "VIX: mean (min to max)",
            f"{data['VIX'].mean():.1f} ({data['VIX'].min():.1f} to {data['VIX'].max():.1f})",
        ),
        (
            f"Days with VIX >= {VIX_STRESS}",
            f"{(data['VIX'] >= VIX_STRESS).mean() * 100:.1f}%",
        ),
    ],
    columns=["Statistic", "Value"],
)
table3.to_csv(f"{OUT}/table3_summary_stats.csv", index=False)
table3

| Expected finding                       | So what                                     | Action                                                   | Leads to           |
| -------------------------------------- | ------------------------------------------- | -------------------------------------------------------- | ------------------ |
| ~1,400 days, few gaps; mean return ≈ 0 | Always guessing "0" is already hard to beat | Use Naive as the bar; score all models by skill vs naive | Tools & evaluation |

**Confirmed from the chart:** _[edit after running: what the chart actually shows]_


## 9. Figure 1: Index price and daily return over time


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(data.index, data["SP500"], color=NAVY, lw=1.3)
axes[0].set_ylabel("S&P 500 level")
title(
    axes[0],
    "Figure 1: The index level trends upward, while daily returns bounce around zero",
)
axes[1].plot(data.index, ret * 100, color=GREY, lw=0.8)
axes[1].axhline(0, color="black", lw=0.6)
axes[1].set_ylabel("Daily return (%)")
for ax in axes:
    style(ax)
fig.text(0.01, -0.01, SOURCE_NOTE, fontsize=8, color="#555")
fig.tight_layout()
fig.savefig(f"{OUT}/fig1_price_and_return.png", dpi=200, bbox_inches="tight")
plt.show()

**Supporting test (ADF):** price should be non-stationary (p > 0.05), returns stationary (p < 0.05).


In [ ]:
adf = pd.DataFrame(
    [
        {
            "Series": "S&P 500 level",
            "ADF stat": adfuller(data["SP500"])[0],
            "p-value": adfuller(data["SP500"])[1],
        },
        {
            "Series": "Daily return",
            "ADF stat": adfuller(ret)[0],
            "p-value": adfuller(ret)[1],
        },
    ]
).round(4)
adf["Stationary?"] = np.where(adf["p-value"] < 0.05, "Yes", "No")
adf

| Expected finding                         | So what                                      | Action                             | Leads to          |
| ---------------------------------------- | -------------------------------------------- | ---------------------------------- | ----------------- |
| Price trends up; returns bounce around 0 | Predicting price looks accurate but misleads | Predict next-day return, not price | Problem statement |

**Confirmed from the chart:** _[edit after running: what the chart actually shows]_


## 10. Figure 2: Correlation of the top 50 stocks with the index


In [ ]:
corr_df = pd.concat([ret.rename("S&P 500"), stock_rets], axis=1)
corr_with_index = stock_rets.corrwith(ret).sort_values(ascending=False)
order = ["S&P 500"] + corr_with_index.index.tolist()
cm = corr_df[order].corr()

fig, ax = plt.subplots(figsize=(11, 9.5))
im = ax.imshow(cm.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(order)), order, rotation=90, fontsize=6.5)
ax.set_yticks(range(len(order)), order, fontsize=6.5)
cbar = fig.colorbar(im, ax=ax, shrink=0.75)
cbar.set_label("Correlation of daily returns")
title(
    ax,
    "Figure 2: Daily-return correlations, index and top 50 stocks (sorted by link to index)",
)
fig.text(0.01, 0.01, SOURCE_NOTE, fontsize=8, color="#555")
fig.tight_layout()
fig.savefig(f"{OUT}/fig2_correlation_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

**Figure 2b (slide-friendly version):** each stock's correlation with the index, one bar per stock.


In [ ]:
c = corr_with_index.sort_values()
fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(c.index, c.values, color=[NAVY if v >= c.median() else LIGHT for v in c.values])
ax.axvline(c.median(), color=ACCENT, ls="--", lw=1)
ax.text(c.median() + 0.01, -1.2, f"median {c.median():.2f}", color=ACCENT, fontsize=8)
ax.tick_params(axis="y", labelsize=8)
ax.set_xlabel("Correlation with S&P 500 daily return")
ax.set_xlim(0, 1)
title(ax, "Figure 2b: How closely each top-50 stock moves with the index", 11)
style(ax)
fig.tight_layout()
fig.savefig(f"{OUT}/fig2b_corr_with_index.png", dpi=200, bbox_inches="tight")
plt.show()

print(
    f"Correlation with index: median {c.median():.2f}, range {c.min():.2f} ({c.idxmin()}) "
    f"to {c.max():.2f} ({c.idxmax()})"
)
print(
    f"Average correlation between pairs of stocks: "
    f"{cm.iloc[1:, 1:].values[np.triu_indices(len(top), 1)].mean():.2f}"
)

| Expected finding                           | So what                                      | Action                                                          | Leads to             |
| ------------------------------------------ | -------------------------------------------- | --------------------------------------------------------------- | -------------------- |
| Largest stocks move closely with the index | The top 50 carry information about the index | Use them as inputs; DA-RNN attention picks the ones that matter | Future work (DA-RNN) |

**Confirmed from the chart:** _[edit after running: what the chart actually shows]_


## 11. Figure 3: VIX vs index returns


In [ ]:
EVENTS = [  # (label, start, end); end=None draws a single line
    ("2022 rate hikes", "2022-03-16", "2022-12-31"),
    ("SVB collapse", "2023-03-10", None),
    ("2025 tariff shock", "2025-04-02", None),
]
lo, hi = data.index.min(), data.index.max()

fig, axes = plt.subplots(
    2, 1, figsize=(11, 6.5), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]}
)
axes[0].plot(data.index, data["VIX"], color=NAVY, lw=1.1)
axes[0].axhline(VIX_STRESS, color=ACCENT, ls="--", lw=1)
axes[0].text(
    lo, VIX_STRESS + 0.8, f"VIX {VIX_STRESS}: stressed", color=ACCENT, fontsize=8
)
axes[0].set_ylabel("VIX")
title(axes[0], "Figure 3: Big index moves cluster when the VIX (market fear) is high")

stressed = data["VIX"] >= VIX_STRESS
axes[1].bar(
    data.index[~stressed],
    ret[~stressed] * 100,
    width=1.5,
    color=GREY,
    label=f"VIX < {VIX_STRESS}",
)
axes[1].bar(
    data.index[stressed],
    ret[stressed] * 100,
    width=1.5,
    color=ACCENT,
    label=f"VIX >= {VIX_STRESS}",
)
axes[1].axhline(0, color="black", lw=0.6)
axes[1].set_ylabel("Index daily return (%)")
axes[1].legend(frameon=False, loc="lower right", fontsize=9)

for label, a, b in EVENTS:
    a = pd.Timestamp(a)
    if not (lo <= a <= hi):
        continue
    for ax in axes:
        if b:
            ax.axvspan(a, min(pd.Timestamp(b), hi), color="#f2e6c9", alpha=0.6, lw=0)
        else:
            ax.axvline(a, color="black", ls=":", lw=1)
    axes[0].text(a, axes[0].get_ylim()[1] * 0.93, " " + label, fontsize=8, va="top")

for ax in axes:
    style(ax)
fig.text(0.01, -0.01, SOURCE_NOTE, fontsize=8, color="#555")
fig.tight_layout()
fig.savefig(f"{OUT}/fig3_vix_vs_returns.png", dpi=200, bbox_inches="tight")
plt.show()

**Figure 3b and calm vs stressed table:** does a higher VIX go with bigger moves?


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.scatter(data["VIX"], ret.abs() * 100, s=8, color=GREY, alpha=0.6)
bins = pd.cut(data["VIX"], bins=[0, 15, 20, 25, 30, 100])
binned = (ret.abs() * 100).groupby(bins, observed=True).mean()
mids = [iv.mid if iv.right < 100 else iv.left + 3 for iv in binned.index]
ax.plot(mids, binned.values, "o-", color=NAVY, lw=2, label="Average size of move")
ax.set_xlabel("VIX on the same day")
ax.set_ylabel("Size of index move, |return| (%)")
title(ax, "Figure 3b: Higher VIX, bigger daily moves", 11)
ax.legend(frameon=False)
style(ax)
fig.tight_layout()
fig.savefig(f"{OUT}/fig3b_vix_scatter.png", dpi=200, bbox_inches="tight")
plt.show()

regimes = pd.DataFrame(
    {
        "Days": [(~stressed).sum(), stressed.sum()],
        "Std of daily return (%)": [
            ret[~stressed].std() * 100,
            ret[stressed].std() * 100,
        ],
        "Days with |move| > 2%": [
            (ret[~stressed].abs() > 0.02).sum(),
            (ret[stressed].abs() > 0.02).sum(),
        ],
    },
    index=[f"Calm (VIX < {VIX_STRESS})", f"Stressed (VIX >= {VIX_STRESS})"],
).round(3)
regimes.to_csv(f"{OUT}/calm_vs_stressed.csv")
regimes

| Expected finding                                                     | So what                                | Action                                                 | Leads to                  |
| -------------------------------------------------------------------- | -------------------------------------- | ------------------------------------------------------ | ------------------------- |
| Big moves cluster when VIX is high (2022 rate hikes, March 2023 SVB) | Price history alone misses market fear | Add VIX and indicators; report calm vs stressed errors | Future work; crisis check |

**Confirmed from the chart:** _[edit after running: what the chart actually shows]_


## 12. Figure 4: Does yesterday's return predict today's? + spread of daily returns


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))

plot_acf(
    ret,
    lags=20,
    zero=False,
    ax=axes[0],
    color=NAVY,
    vlines_kwargs={"colors": NAVY},
    title=None,
)
for coll in axes[0].collections:
    if isinstance(coll, plt.matplotlib.collections.PolyCollection):
        coll.set_facecolor(LIGHT)
axes[0].set_xlabel("Lag (trading days)")
axes[0].set_ylabel("Correlation with today's return")
title(axes[0], "Figure 4a: Past returns barely predict the next", 11)

mu, sd = ret.mean() * 100, ret.std() * 100
axes[1].hist(ret * 100, bins=80, color=GREY, density=True, label="Actual returns")
x = np.linspace(ret.min() * 100, ret.max() * 100, 300)
axes[1].plot(
    x,
    stats.norm.pdf(x, mu, sd),
    color=NAVY,
    lw=1.8,
    label="Normal curve, same mean/std",
)
extreme = ret.abs() * 100 > 3 * sd
axes[1].set_xlabel("Daily return (%)")
axes[1].set_ylabel("Density")
title(axes[1], "Figure 4b: Mostly small moves, plus a few extreme days", 11)
axes[1].legend(frameon=False, fontsize=8)
axes[1].text(
    0.02,
    0.95,
    f"Excess kurtosis: {stats.kurtosis(ret):.1f}\n"
    f"Days beyond ±3 std: {extreme.sum()}\n(a normal curve expects ~{0.0027 * len(ret):.0f})",
    transform=axes[1].transAxes,
    fontsize=8,
    va="top",
)

for ax in axes:
    style(ax)
fig.tight_layout()
fig.savefig(f"{OUT}/fig4_autocorr_and_histogram.png", dpi=200, bbox_inches="tight")
plt.show()

lb = acorr_ljungbox(ret, lags=[1, 5, 10], return_df=True).round(4)
lb.index.name = "lag"
print("Ljung-Box test (p < 0.05 = some predictable pattern in past returns):")
display(lb)
print("Most extreme days:")
display((ret[extreme] * 100).round(2).sort_values().rename("return (%)").to_frame())

| Expected finding                                         | So what                                                           | Action                                                                     | Leads to            |
| -------------------------------------------------------- | ----------------------------------------------------------------- | -------------------------------------------------------------------------- | ------------------- |
| Past returns barely predict the next; a few extreme days | ARIMA will likely score close to naive; RMSE is hit by crash days | Report MAE too; move to models that learn complex patterns (XGBoost, LSTM) | Preliminary results |

**Confirmed from the chart:** _[edit after running: what the chart actually shows]_


## 13. EDA takeaway

Returns look random (Table 3, Figures 1 and 4) → so beating naive is the real test → but big stocks and market fear carry signal (Figures 2 and 3) → so later models add those inputs.

_[Edit after running: note anything the charts did not confirm.]_


In [ ]:
sorted(os.listdir(OUT))